# TensorFlowLinear CPU training

A single linear layer learns y = 2x + 1. Data preparation, model construction, training, evaluation and checkpoint restoration use the public Bovi contracts.

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4

from bovi_core.config import Config
from bovi_core.ml import EvaluationContext, ResolvedCheckpoint, TrainingContext
from tensorflow_linear import (
    TensorFlowLinearEvaluationConfig,
    TensorFlowLinearEvaluator,
    TensorFlowLinearModelConfig,
    TensorFlowLinearModelProvider,
    TensorFlowLinearTrainer,
    TensorFlowLinearTrainingConfig,
    create_dataloader,
)

Config.reset()
config = Config(experiment_name="tensorflow_linear", project_name="tensorflow-linear")
model_config = TensorFlowLinearModelConfig.from_config(config)
training_config = TensorFlowLinearTrainingConfig.from_config(config)
loaders = {
    split: create_dataloader(config, model_config, split) for split in ("train", "validation")
}
print(model_config)
print(next(iter(loaders["train"])))

I0000 00:00:1788950532.326841  115859 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788950532.625117  115859 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1788950536.935688  115859 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788950536.937414  115859 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


🔄 Config singleton reset
🔧 Initializing Config...
   🌍 Environment: local
🔍 No TOML path passed, resolving path automatically
🔍 Found pyproject.toml in parent directory: /home/douwe/work/code/bovi-analytics/bovi/.worktrees/model-package-layout/packages/models/tensorflow-linear
      📋 Project name: tensorflow-linear
      📁 Current working directory: /home/douwe/work/code/bovi-analytics/bovi/.worktrees/model-package-layout/packages/models/tensorflow-linear/notebooks/experiments/tensorflow_linear
      📁 Traversed to: /home/douwe/work/code/bovi-analytics/bovi/.worktrees/model-package-layout/packages/models/tensorflow-linear
      📁 project_src: /home/douwe/work/code/bovi-analytics/bovi/.worktrees/model-package-layout/packages/models/tensorflow-linear/src
⚠️  .env file not specified or not found at: /home/douwe/work/code/bovi-analytics/bovi/.worktrees/model-package-layout/packages/models/tensorflow-linear/.env
   👤 Author: Douwe de Kok <douwedekok@gmail.com>
🔍 No config_file_path passed,

In [2]:
workspace = TemporaryDirectory(prefix="tensorflow-linear-")
output = Path(workspace.name)
provider = TensorFlowLinearModelProvider()
model = provider.create(model_config)
context = TrainingContext(run_id=uuid4(), output_dir=output / "first")
result = TensorFlowLinearTrainer(model, loaders, training_config, context).train()
assert result.status == "completed", result.issues
print(result.status, result.stop_reason)
print("First epoch:", result.epochs[0].metrics)
print("Last epoch:", result.epochs[-1].metrics)
assert result.epochs[-1].metrics["train_mse"] < 0.001
print("Best epoch:", result.best_epoch)

E0000 00:00:1788950542.648967  115859 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


completed max_epochs_reached
First epoch: {'train_mse': 1.5945777893066406, 'train_mae': 1.063295841217041, 'validation_mse': 1.5450897216796875, 'validation_mae': 1.0299804210662842}
Last epoch: {'train_mse': 5.8121297996649446e-08, 'train_mae': 0.0002188161015510559, 'validation_mse': 5.580354311973679e-08, 'validation_mae': 0.0002100244164466858}
Best epoch: 40


In [3]:
evaluation = TensorFlowLinearEvaluator(
    model, TensorFlowLinearEvaluationConfig.from_config(config)
).evaluate(
    loaders["validation"],
    EvaluationContext(
        evaluation_id=uuid4(),
        split="validation",
        model_version="last",
        training_run_id=context.run_id,
        output_dir=output / "evaluation",
    ),
)
assert evaluation.status == "completed", evaluation.issues
print(evaluation.metrics)
print("Prediction at x=0.5 (expected 2):", model([[0.5]]))

{'mse': 5.580354311973679e-08, 'mae': 0.0002100244164466858}
Prediction at x=0.5 (expected 2): [1.9998507]


## Resume

Load the last checkpoint into a new model. A new attempt gets its own run ID and starts at epoch 1. Best and last checkpoints stay on disk, while results contain references.

In [4]:
reference = result.last_checkpoint
restored = provider.restore_checkpoint(
    model_config,
    ResolvedCheckpoint(
        format=reference.format,
        source_uri=reference.uri,
        local_path=context.output_dir / "checkpoints" / "last.keras",
    ),
)
resume_context = TrainingContext(
    run_id=uuid4(), resumed_from_run_id=context.run_id, output_dir=output / "resume"
)
resumed = TensorFlowLinearTrainer(
    restored, loaders, TensorFlowLinearTrainingConfig(epochs=2), resume_context
).train()
assert resumed.status == "completed", resumed.issues
assert resumed.epochs[0].epoch == 1
print(resumed.epochs[-1].metrics)
workspace.cleanup()
Config.reset()

{'train_mse': 2.4412553756292255e-08, 'train_mae': 0.00014182180166244507, 'validation_mse': 2.346164151845187e-08, 'validation_mae': 0.0001361742615699768}
🔄 Config singleton reset
